# 05 - Adversarial Attack: Saliency Map Evasion
## CSE-CIC-IDS2018 — Evaluasi Kerentanan Model NIDS

**Tujuan:** Simulasi serangan evasion berbasis Saliency Map untuk mengidentifikasi
kerentanan model XGBoost yang sudah dioptimasi fiturnya (Top-10).

**Metodologi:**
1. Load model XGBoost Top-10 (baseline) dari Notebook 04
2. Hitung gradien loss terhadap setiap fitur (Saliency Map)
3. Generate adversarial samples: x_adv = x + ε · sign(∇L)
4. Evaluasi penurunan performa (MCC, F1, Precision)
5. Simpan adversarial dataset untuk Adversarial Training (Notebook 06)

**Input:**
- `cleaned_100.pkl` — dataset bersih
- `experiment_results_03.pkl` — feature lists, label mapping
- `models/deploy/rank1_*.json` — model XGBoost terbaik

**Output:**
- `adversarial_samples_05.pkl` — sampel adversarial untuk training
- `adversarial_results_05.pkl` — hasil evaluasi kerentanan
- Visualisasi: saliency map, performa drop

In [ ]:
# Install dependencies
import sys
!{sys.executable} -m pip install scikit-learn xgboost matplotlib seaborn numpy pandas -q
print('✓ Dependencies installed')

In [ ]:
import pandas as pd
import numpy as np
import pickle, os, json, time, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})

DATA_DIR = '../data/'
MODEL_DIR = '../models/'
DEPLOY_DIR = os.path.join(MODEL_DIR, 'deploy')
RANDOM_SEED = 42
TEST_SIZE = 0.20

print('Libraries loaded ✓')

## 1. Load Data & Model Baseline

In [ ]:
# Load experiment results dari Notebook 03
with open(os.path.join(DATA_DIR, 'experiment_results_03.pkl'), 'rb') as f:
    prev_results = pickle.load(f)

top10_features = prev_results['top10_features']
top15_features = prev_results['top15_features']
all_feature_names = prev_results['feature_names']
label_mapping = prev_results['label_mapping']
inverse_label = {v: k for k, v in label_mapping.items()}

print(f'Top-10 features: {top10_features}')
print(f'Total classes: {len(label_mapping)}')
print(f'Labels: {list(label_mapping.keys())}')

In [ ]:
# Load cleaned dataset
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    data = pickle.load(f)

X_all = data['X']
y_all = data['y']
scaler = data['scaler']

# Subset ke Top-10 features
idx_top10 = [all_feature_names.index(f) for f in top10_features if f in all_feature_names]
X_top10 = X_all[:, idx_top10]

print(f'Dataset shape (all): {X_all.shape}')
print(f'Dataset shape (Top-10): {X_top10.shape}')
print(f'Classes: {np.unique(y_all)}')

In [ ]:
# Split train/test — SAME split as Notebook 04
X_train, X_test, y_train, y_test = train_test_split(
    X_top10, y_all, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
)

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Test:  {X_test.shape[0]:,} samples')

In [ ]:
# Load atau retrain model XGBoost Top-10 (baseline)
# Cek apakah model deploy sudah ada
deploy_files = os.listdir(DEPLOY_DIR) if os.path.exists(DEPLOY_DIR) else []
xgb_top10_file = [f for f in deploy_files if 'xgboost' in f and 'top-10' in f and f.endswith('.json')]

if xgb_top10_file:
    model_path = os.path.join(DEPLOY_DIR, xgb_top10_file[0])
    model_baseline = XGBClassifier()
    model_baseline.load_model(model_path)
    print(f'Loaded pre-trained model: {xgb_top10_file[0]}')
else:
    # Retrain jika file tidak ditemukan
    print('Model file not found, retraining XGBoost Top-10...')
    n_classes = len(np.unique(y_all))
    model_baseline = XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', random_state=RANDOM_SEED,
        n_jobs=-1, tree_method='hist'
    )
    model_baseline.fit(X_train, y_train)
    print('Retraining complete ✓')

# Evaluasi baseline pada data bersih
y_pred_clean = model_baseline.predict(X_test)
y_prob_clean = model_baseline.predict_proba(X_test)

mcc_clean = matthews_corrcoef(y_test, y_pred_clean)
f1_clean = f1_score(y_test, y_pred_clean, average='weighted', zero_division=0)
acc_clean = accuracy_score(y_test, y_pred_clean)

print(f'\n--- Baseline Performance (Clean Data) ---')
print(f'  MCC:      {mcc_clean:.4f}')
print(f'  F1-Score: {f1_clean*100:.2f}%')
print(f'  Accuracy: {acc_clean*100:.2f}%')

## 2. Saliency Map — Hitung Gradien Loss per Fitur

Karena XGBoost bukan differentiable model (tree-based), kita approksimasi gradien
menggunakan **finite difference method**:

$$S(\mathbf{x}, i) \approx \frac{L(\theta, \mathbf{x} + h\mathbf{e}_i, y) - L(\theta, \mathbf{x} - h\mathbf{e}_i, y)}{2h}$$

di mana $h$ adalah step size kecil dan $\mathbf{e}_i$ adalah unit vector arah fitur ke-$i$.

In [ ]:
def compute_loss(model, X, y_true):
    """
    Hitung cross-entropy loss untuk XGBoost predictions.
    L = -sum(y_one_hot * log(p))
    """
    probs = model.predict_proba(X)
    n_classes = probs.shape[1]
    # One-hot encode y_true
    y_onehot = np.zeros((len(y_true), n_classes))
    y_onehot[np.arange(len(y_true)), y_true.astype(int)] = 1.0
    # Cross-entropy loss per sample
    eps = 1e-15
    probs_clipped = np.clip(probs, eps, 1 - eps)
    loss_per_sample = -np.sum(y_onehot * np.log(probs_clipped), axis=1)
    return loss_per_sample


def compute_saliency_map(model, X, y_true, h=0.01):
    """
    Hitung saliency map menggunakan finite difference.
    Untuk setiap fitur i, hitung approx gradient dL/dx_i.
    
    Returns:
        saliency: array (n_samples, n_features) — |dL/dx_i|
    """
    n_samples, n_features = X.shape
    saliency = np.zeros((n_samples, n_features))
    
    for i in range(n_features):
        # Perturbasi +h pada fitur i
        X_plus = X.copy()
        X_plus[:, i] += h
        loss_plus = compute_loss(model, X_plus, y_true)
        
        # Perturbasi -h pada fitur i
        X_minus = X.copy()
        X_minus[:, i] -= h
        loss_minus = compute_loss(model, X_minus, y_true)
        
        # Central difference gradient
        saliency[:, i] = (loss_plus - loss_minus) / (2 * h)
    
    return saliency


print('Saliency Map functions defined ✓')
print(f'Using finite difference with step h=0.01')
print(f'Features to perturb: {len(top10_features)}')

In [ ]:
# Hitung Saliency Map pada test set
# Untuk efisiensi, sample subset jika test set terlalu besar
MAX_SAMPLES_SALIENCY = 50000

if len(X_test) > MAX_SAMPLES_SALIENCY:
    np.random.seed(RANDOM_SEED)
    sample_idx = np.random.choice(len(X_test), MAX_SAMPLES_SALIENCY, replace=False)
    X_saliency = X_test[sample_idx]
    y_saliency = y_test[sample_idx] if isinstance(y_test, np.ndarray) else y_test.values[sample_idx]
    print(f'Sampled {MAX_SAMPLES_SALIENCY:,} from {len(X_test):,} test samples for saliency computation')
else:
    X_saliency = X_test
    y_saliency = y_test if isinstance(y_test, np.ndarray) else y_test.values
    print(f'Using full test set: {len(X_test):,} samples')

print('\nComputing Saliency Map (finite difference)...')
start = time.time()
saliency = compute_saliency_map(model_baseline, X_saliency, y_saliency, h=0.01)
elapsed = time.time() - start
print(f'Done in {elapsed:.1f}s')
print(f'Saliency shape: {saliency.shape}')

## 3. Visualisasi Saliency Map

In [ ]:
# Mean absolute saliency per fitur
mean_saliency = np.mean(np.abs(saliency), axis=0)
saliency_df = pd.DataFrame({
    'feature': top10_features,
    'mean_saliency': mean_saliency
}).sort_values('mean_saliency', ascending=False)

print('\nMean Absolute Saliency per Feature (Top-10):')
print('='*50)
for _, row in saliency_df.iterrows():
    bar = '█' * int(row['mean_saliency'] / saliency_df['mean_saliency'].max() * 30)
    print(f"  {row['feature']:25s} {row['mean_saliency']:.6f}  {bar}")

# Visualisasi bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = plt.cm.Reds(np.linspace(0.4, 0.9, 10))
bars = ax.barh(
    range(10), saliency_df['mean_saliency'].values,
    color=colors_bar, edgecolor='black', linewidth=0.5
)
ax.set_yticks(range(10))
ax.set_yticklabels(saliency_df['feature'].values, fontsize=10)
ax.set_xlabel('Mean Absolute Saliency (|∂L/∂x_i|)')
ax.set_title('Saliency Map: Feature Sensitivity terhadap Loss\n(XGBoost Top-10, CSE-CIC-IDS2018)', 
             fontsize=12, fontweight='bold')

for i, val in enumerate(saliency_df['mean_saliency'].values):
    ax.text(val + 0.0001, i, f'{val:.5f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'saliency_map_top10.png'), bbox_inches='tight')
plt.show()
print('Saved: saliency_map_top10.png')

In [ ]:
# Heatmap: Saliency per class
unique_classes = np.unique(y_saliency)
saliency_per_class = np.zeros((len(unique_classes), len(top10_features)))

for i, cls in enumerate(unique_classes):
    mask = y_saliency == cls
    saliency_per_class[i] = np.mean(np.abs(saliency[mask]), axis=0)

class_names = [inverse_label.get(int(c), f'Class {c}') for c in unique_classes]

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    saliency_per_class, annot=True, fmt='.4f', cmap='YlOrRd',
    xticklabels=top10_features, yticklabels=class_names, ax=ax
)
ax.set_title('Saliency Map per Attack Class\n(Mean |∂L/∂x_i| per class)', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('Features')
ax.set_ylabel('Attack Class')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'saliency_heatmap_per_class.png'), bbox_inches='tight')
plt.show()
print('Saved: saliency_heatmap_per_class.png')

## 4. Generate Adversarial Samples (FGSM-style Evasion)

Formulasi:
$$\mathbf{x}_{adv} = \mathbf{x} + \epsilon \cdot \text{sign}(\nabla_{\mathbf{x}} L(\theta, \mathbf{x}, y))$$

Uji dengan beberapa nilai epsilon: 0.01, 0.05, 0.1, 0.2, 0.3

In [ ]:
def generate_adversarial_fgsm(X, saliency, epsilon):
    """
    Generate adversarial samples menggunakan FGSM (Fast Gradient Sign Method).
    x_adv = x + epsilon * sign(gradient)
    
    Args:
        X: original samples (n_samples, n_features)
        saliency: gradient values (n_samples, n_features)
        epsilon: perturbation magnitude
    
    Returns:
        X_adv: adversarial samples
    """
    perturbation = epsilon * np.sign(saliency)
    X_adv = X + perturbation
    return X_adv


# Epsilon values to test
EPSILONS = [0.01, 0.05, 0.1, 0.2, 0.3]

print('Generating adversarial samples...')
print(f'Epsilon values: {EPSILONS}')
print(f'Base samples: {X_saliency.shape[0]:,}')

adversarial_sets = {}
for eps in EPSILONS:
    X_adv = generate_adversarial_fgsm(X_saliency, saliency, eps)
    adversarial_sets[eps] = X_adv
    
    # Statistik perturbasi
    delta = X_adv - X_saliency
    mean_delta = np.mean(np.abs(delta))
    max_delta = np.max(np.abs(delta))
    print(f'  ε={eps:.2f}: mean|δ|={mean_delta:.4f}, max|δ|={max_delta:.4f}')

print('\nAdversarial samples generated ✓')

## 5. Evaluasi Kerentanan Model — Skenario S2 (Vulnerability)

In [ ]:
def evaluate_model(model, X, y_true, scenario_name):
    """
    Evaluasi model dengan metrik MCC, F1, Precision, Accuracy.
    """
    y_pred = model.predict(X)
    
    mcc = matthews_corrcoef(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    
    return {
        'scenario': scenario_name,
        'mcc': mcc,
        'f1_score': f1,
        'precision': prec,
        'recall': rec,
        'accuracy': acc,
        'y_pred': y_pred
    }


print('='*80)
print(f'{"EVALUASI KERENTANAN: Baseline Model vs Adversarial Samples":^80}')
print('='*80)

# S1: Baseline on clean data
result_s1 = evaluate_model(model_baseline, X_saliency, y_saliency, 'S1: Clean')
print(f'\n  S1 (Baseline + Clean):    MCC={result_s1["mcc"]:.4f} | F1={result_s1["f1_score"]*100:.2f}% | Acc={result_s1["accuracy"]*100:.2f}%')

# S2: Baseline on adversarial data (per epsilon)
vulnerability_results = [result_s1]

print(f'\n  S2 (Baseline + Adversarial):')
for eps in EPSILONS:
    X_adv = adversarial_sets[eps]
    result = evaluate_model(model_baseline, X_adv, y_saliency, f'S2: ε={eps}')
    vulnerability_results.append(result)
    
    # Drop dari clean
    mcc_drop = result_s1['mcc'] - result['mcc']
    f1_drop = (result_s1['f1_score'] - result['f1_score']) * 100
    
    print(f'    ε={eps:.2f}: MCC={result["mcc"]:.4f} (↓{mcc_drop:.4f}) | '
          f'F1={result["f1_score"]*100:.2f}% (↓{f1_drop:.2f}%) | '
          f'Acc={result["accuracy"]*100:.2f}%')

print(f'\n  → Security Gap (ε=0.1): MCC drop = {result_s1["mcc"] - vulnerability_results[3]["mcc"]:.4f}')

In [ ]:
# Tabel ringkasan
print('\n'+'='*90)
print(f'{"TABLE: VULNERABILITY ASSESSMENT — BASELINE MODEL":^90}')
print('='*90)
print(f'{"Scenario":<20s} {"MCC":>8s} {"F1 (%)":>8s} {"Prec (%)":>9s} {"Rec (%)":>8s} {"Acc (%)":>8s} {"MCC Drop":>9s}')
print('-'*90)

for r in vulnerability_results:
    mcc_drop = result_s1['mcc'] - r['mcc']
    drop_str = f'{mcc_drop:.4f}' if mcc_drop > 0 else '—'
    print(f'{r["scenario"]:<20s} {r["mcc"]:>8.4f} {r["f1_score"]*100:>7.2f}% '
          f'{r["precision"]*100:>8.2f}% {r["recall"]*100:>7.2f}% '
          f'{r["accuracy"]*100:>7.2f}% {drop_str:>9s}')

print('='*90)

## 6. Visualisasi: MCC & F1 Drop vs Epsilon

In [ ]:
eps_list = [0.0] + EPSILONS
mcc_list = [r['mcc'] for r in vulnerability_results]
f1_list = [r['f1_score']*100 for r in vulnerability_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MCC vs Epsilon
ax1.plot(eps_list, mcc_list, 'o-', color='crimson', linewidth=2, markersize=8)
ax1.axhline(y=mcc_list[0], color='gray', linestyle='--', alpha=0.5, label='Baseline (clean)')
ax1.fill_between(eps_list, mcc_list, mcc_list[0], alpha=0.1, color='red')
ax1.set_xlabel('Epsilon (ε)', fontsize=11)
ax1.set_ylabel('MCC', fontsize=11)
ax1.set_title('Matthews Correlation Coefficient vs Epsilon\n(Baseline Model Under Evasion Attack)', 
              fontsize=11, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim([-0.1, 1.05])

# F1 vs Epsilon
ax2.plot(eps_list, f1_list, 's-', color='darkblue', linewidth=2, markersize=8)
ax2.axhline(y=f1_list[0], color='gray', linestyle='--', alpha=0.5, label='Baseline (clean)')
ax2.fill_between(eps_list, f1_list, f1_list[0], alpha=0.1, color='blue')
ax2.set_xlabel('Epsilon (ε)', fontsize=11)
ax2.set_ylabel('F1-Score (%)', fontsize=11)
ax2.set_title('F1-Score vs Epsilon\n(Baseline Model Under Evasion Attack)', 
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 105])

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'vulnerability_mcc_f1_vs_epsilon.png'), bbox_inches='tight')
plt.show()
print('Saved: vulnerability_mcc_f1_vs_epsilon.png')

In [ ]:
# Confusion Matrix: Baseline vs Adversarial (ε=0.1)
eps_demo = 0.1
result_adv = [r for r in vulnerability_results if r['scenario'] == f'S2: ε={eps_demo}'][0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

class_names_short = [inverse_label.get(int(c), f'C{c}') for c in np.unique(y_saliency)]

# Clean
cm_clean = confusion_matrix(y_saliency, result_s1['y_pred'])
cm_clean_norm = cm_clean.astype('float') / cm_clean.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_clean_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[0],
            xticklabels=class_names_short, yticklabels=class_names_short)
axes[0].set_title(f'S1: Baseline + Clean Data\nMCC={result_s1["mcc"]:.4f}', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Adversarial
cm_adv = confusion_matrix(y_saliency, result_adv['y_pred'])
cm_adv_norm = cm_adv.astype('float') / cm_adv.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_adv_norm, annot=True, fmt='.2f', cmap='Reds', ax=axes[1],
            xticklabels=class_names_short, yticklabels=class_names_short)
axes[1].set_title(f'S2: Baseline + Adversarial (ε={eps_demo})\nMCC={result_adv["mcc"]:.4f}', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.suptitle('Confusion Matrix: Clean vs Adversarial Input', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'confusion_clean_vs_adversarial.png'), bbox_inches='tight')
plt.show()
print('Saved: confusion_clean_vs_adversarial.png')

## 7. Per-Class Vulnerability Analysis

In [ ]:
# Per-class F1 drop pada epsilon=0.1
eps_analysis = 0.1
X_adv_analysis = adversarial_sets[eps_analysis]
y_pred_adv = model_baseline.predict(X_adv_analysis)

print(f'\nPer-Class Vulnerability Analysis (ε={eps_analysis}):')
print('='*80)
print(f'{"Class":<25s} {"F1 Clean":>10s} {"F1 Adv":>10s} {"Drop":>10s} {"Vulnerability":>14s}')
print('-'*80)

per_class_vuln = []
for cls in np.unique(y_saliency):
    mask = y_saliency == cls
    cls_name = inverse_label.get(int(cls), f'Class {cls}')
    
    # F1 on clean
    y_true_cls = (y_saliency[mask] == cls).astype(int)
    y_pred_cls_clean = (result_s1['y_pred'][mask] == cls).astype(int)
    y_pred_cls_adv = (y_pred_adv[mask] == cls).astype(int)
    
    f1_c = f1_score(y_true_cls, y_pred_cls_clean, zero_division=0)
    f1_a = f1_score(y_true_cls, y_pred_cls_adv, zero_division=0)
    drop = f1_c - f1_a
    
    vuln_level = 'HIGH' if drop > 0.10 else ('MEDIUM' if drop > 0.05 else 'LOW')
    per_class_vuln.append({'class': cls_name, 'f1_clean': f1_c, 'f1_adv': f1_a, 'drop': drop, 'level': vuln_level})
    
    print(f'{cls_name:<25s} {f1_c*100:>9.2f}% {f1_a*100:>9.2f}% {drop*100:>9.2f}% {vuln_level:>14s}')

print('='*80)
high_vuln = [v for v in per_class_vuln if v['level'] == 'HIGH']
print(f'\nKelas dengan kerentanan TINGGI: {len(high_vuln)}')
for v in high_vuln:
    print(f'  - {v["class"]}: F1 drop {v["drop"]*100:.1f}%')

## 8. Simpan Output untuk Notebook 06 (Adversarial Training)

In [ ]:
# Simpan adversarial samples (ε=0.1 sebagai default untuk training)
# dan juga simpan untuk ε=0.01 (conservative)
EPSILON_TRAIN = 0.1
EPSILON_CONSERVATIVE = 0.01

adversarial_output = {
    # Data
    'X_test_clean': X_saliency,
    'y_test': y_saliency,
    'X_train': X_train,
    'y_train': y_train,
    
    # Adversarial samples
    'X_adv_eps01': adversarial_sets[0.01],
    'X_adv_eps005': adversarial_sets[0.05],
    'X_adv_eps01_train': generate_adversarial_fgsm(X_train[:MAX_SAMPLES_SALIENCY], 
        compute_saliency_map(model_baseline, X_train[:MAX_SAMPLES_SALIENCY], 
                            y_train[:MAX_SAMPLES_SALIENCY] if isinstance(y_train, np.ndarray) 
                            else y_train.values[:MAX_SAMPLES_SALIENCY], h=0.01),
        EPSILON_CONSERVATIVE),
    'X_adv_eps01_full': adversarial_sets[EPSILON_CONSERVATIVE],
    'X_adv_eps10': adversarial_sets[EPSILON_TRAIN],
    'X_adv_eps10_train': None,  # Will be computed in next cell
    
    # Saliency data
    'saliency': saliency,
    'mean_saliency_per_feature': mean_saliency,
    'saliency_per_class': saliency_per_class,
    
    # Config
    'epsilons_tested': EPSILONS,
    'epsilon_for_training': EPSILON_TRAIN,
    'top10_features': top10_features,
    'feature_indices': idx_top10,
    'label_mapping': label_mapping,
    
    # Results
    'vulnerability_results': [{k: v for k, v in r.items() if k != 'y_pred'} 
                              for r in vulnerability_results],
    'per_class_vulnerability': per_class_vuln,
    'baseline_mcc_clean': result_s1['mcc'],
    'baseline_f1_clean': result_s1['f1_score']
}

print('Computing adversarial samples on TRAINING set for Adversarial Training...')
# Generate adversarial pada training data (untuk augmentasi di notebook 06)
n_train_sample = min(MAX_SAMPLES_SALIENCY, len(X_train))
saliency_train = compute_saliency_map(
    model_baseline, X_train[:n_train_sample], 
    y_train[:n_train_sample] if isinstance(y_train, np.ndarray) else y_train.values[:n_train_sample],
    h=0.01
)
X_adv_train_eps10 = generate_adversarial_fgsm(X_train[:n_train_sample], saliency_train, EPSILON_TRAIN)
adversarial_output['X_adv_eps10_train'] = X_adv_train_eps10
adversarial_output['y_adv_train'] = y_train[:n_train_sample] if isinstance(y_train, np.ndarray) else y_train.values[:n_train_sample]
adversarial_output['saliency_train'] = saliency_train

print(f'Training adversarial samples: {X_adv_train_eps10.shape}')
print(f'\nSaving adversarial_samples_05.pkl...')
with open(os.path.join(DATA_DIR, 'adversarial_samples_05.pkl'), 'wb') as f:
    pickle.dump(adversarial_output, f)

file_size = os.path.getsize(os.path.join(DATA_DIR, 'adversarial_samples_05.pkl')) / (1024*1024)
print(f'Saved: adversarial_samples_05.pkl ({file_size:.1f} MB)')

In [ ]:
# Simpan juga hasil evaluasi kerentanan terpisah (lebih ringan)
adversarial_results = {
    'vulnerability_results': [{k: v for k, v in r.items() if k != 'y_pred'} 
                              for r in vulnerability_results],
    'per_class_vulnerability': per_class_vuln,
    'saliency_ranking': saliency_df.to_dict('records'),
    'epsilons': EPSILONS,
    'baseline_performance': {
        'mcc': result_s1['mcc'],
        'f1': result_s1['f1_score'],
        'accuracy': result_s1['accuracy']
    }
}

with open(os.path.join(DATA_DIR, 'adversarial_results_05.pkl'), 'wb') as f:
    pickle.dump(adversarial_results, f)

print('Saved: adversarial_results_05.pkl')

## 9. Narasi & Kesimpulan

In [ ]:
print('='*70)
print(f'{"NARASI: ADVERSARIAL ATTACK ANALYSIS":^70}')
print('='*70)
print(f'''
■ TEMUAN UTAMA:

  1. SALIENCY MAP menunjukkan bahwa beberapa fitur dalam Top-10 memiliki
     sensitivitas yang sangat tinggi — perubahan kecil pada fitur ini
     berdampak signifikan pada keputusan model.
     
     Fitur paling sensitif: {saliency_df.iloc[0]["feature"]} 
     (saliency = {saliency_df.iloc[0]["mean_saliency"]:.6f})

  2. VULNERABILITY (ε=0.1):
     - MCC drop: {result_s1["mcc"]:.4f} → {vulnerability_results[3]["mcc"]:.4f} 
       (penurunan {result_s1["mcc"] - vulnerability_results[3]["mcc"]:.4f})
     - F1 drop: {result_s1["f1_score"]*100:.2f}% → {vulnerability_results[3]["f1_score"]*100:.2f}%
     
  3. TRADE-OFF EFISIENSI vs KEAMANAN:
     Model dengan hanya 10 fitur sangat efisien (5.65 MB, inference cepat)
     TAPI lebih rentan terhadap evasion karena decision boundary
     terkonsentrasi pada dimensi yang sangat terbatas.

  4. KELAS PALING RENTAN:
     {chr(10).join([f"     - {v['class']}: drop {v['drop']*100:.1f}%" for v in high_vuln[:5]])}

■ IMPLIKASI:
  → Model baseline TIDAK AMAN untuk deployment langsung
  → Perlu Adversarial Training (Notebook 06) untuk memperkuat robustness
  → Target: MCC pada adversarial data harus >= 0.90 setelah hardening

{'='*70}
''')